In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [4]:
missing_count = df.isnull().sum()

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_data = pd.DataFrame({
    'Missing Values': missing_count,
    'Percentage (%)': missing_percentage
})
missing_data

,Missing Values,Percentage (%)
InvoiceNo,0,0.000000
StockCode,0,0.000000
Description,1454,0.268311
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
CustomerID,135080,24.926694
Country,0,0.000000


In [5]:
unique_counts = df.nunique()

unique_summary = pd.DataFrame({
    'Unique Values': unique_counts
})
unique_summary

,Unique Values
InvoiceNo,25900
StockCode,4070
Description,4223
Quantity,722
InvoiceDate,23260
UnitPrice,1630
CustomerID,4372
Country,38


In [6]:
df["missing_customer"] = df["CustomerID"].isna().astype(int)

In [7]:
group_stats = df.groupby("missing_customer").agg({
    "Quantity": ["mean","median","std"],
    "UnitPrice": ["mean","median","std"],
})

group_stats

Quantity                    UnitPrice                   
                       mean median         std      mean median         std
missing_customer                                                           
0                 12.061303    5.0  248.693370  3.460471   1.95   69.315162
1                  1.995573    1.0   66.696153  8.076577   3.29  151.900816

In [8]:
df.groupby("missing_customer").agg({
    "Quantity": "mean",
    "UnitPrice": "mean"
})

,Quantity,UnitPrice
missing_customer,,
0,12.061303,3.460471
1,1.995573,8.076577


In [9]:
df["luxury_purchase"] = (
    (df["UnitPrice"] > df["UnitPrice"].quantile(0.9)) &
    (df["Quantity"] <= 2)
).astype(int)

df.groupby("missing_customer")["luxury_purchase"].mean()

missing_customer
0    0.053671
1    0.130841
Name: luxury_purchase, dtype: float64

The missingness of CustomerID is most consistent with Missing At Random (MAR).
Rows with missing CustomerID differ systematically from rows where CustomerID is present. Transactions without CustomerID have significantly lower average quantities (1.99 vs 12.06), higher average unit prices (8.07 vs 3.46), and a higher proportion of luxury purchases (13.08% vs 5.37%). Since the probability of missing CustomerID appears to depend on other observed variables such as price and quantity, the missingness is not completely random. Therefore, the missingness mechanism is best classified as MAR.

In [10]:
df[df['Description'].isna()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer,luxury_purchase
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,1,0
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,1,0
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,1,0
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,1,0
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,1,0
...,...,...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom,1,0
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom,1,0
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom,1,0
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,1,0


In [11]:
df[df['Description'].isna()]['UnitPrice'].sum()

np.float64(0.0)

The missingness of the Description field is classified as MAR. My analysis shows that the probability of a description being missing is perfectly predictable based on the UnitPrice variable (where $Price = 0$). These records represent administrative adjustments, damaged stock entries, or internal logs rather than consumer purchases.

In [12]:
df[df['CustomerID'].isna()]['Country'].value_counts()

Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
France                66
Israel                47
Portugal              39
Bahrain                2
Name: count, dtype: int64

In [13]:
df['Country'].value_counts()

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [14]:
missing_desc_mask = df['Description'].isnull()
missing_desc_df = df[missing_desc_mask]

In [15]:
valid_stock_codes = df.dropna(subset=['Description'])['StockCode'].unique()
missing_with_known_code = missing_desc_df[missing_desc_df['StockCode'].isin(valid_stock_codes)]
missing_with_known_code

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer,luxury_purchase
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,1,0
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,1,0
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,1,0
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom,1,0
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,1,0
...,...,...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom,1,0
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom,1,0
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom,1,0
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,1,0


In [16]:
df[df['StockCode'] == 22139]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer,luxury_purchase
106,536381,22139,RETROSPOT TEA SET CERAMIC 11 PC,23,2010-12-01 09:41:00,4.25,15311.0,United Kingdom,0,0
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom,1,0
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.00,NaN,United Kingdom,1,0
6885,536982,22139,RETROSPOT TEA SET CERAMIC 11 PC,10,2010-12-03 14:27:00,11.02,NaN,United Kingdom,1,0
7203,537011,22139,NaN,-5,2010-12-03 15:38:00,0.00,NaN,United Kingdom,1,0
...,...,...,...,...,...,...,...,...,...,...
538411,581405,22139,RETROSPOT TEA SET CERAMIC 11 PC,1,2011-12-08 13:50:00,4.95,13521.0,United Kingdom,0,0
539531,581439,22139,RETROSPOT TEA SET CERAMIC 11 PC,1,2011-12-08 16:30:00,10.79,NaN,United Kingdom,1,1
540441,581486,22139,RETROSPOT TEA SET CERAMIC 11 PC,6,2011-12-09 09:38:00,4.95,17001.0,United Kingdom,0,0
541387,581498,22139,RETROSPOT TEA SET CERAMIC 11 PC,2,2011-12-09 10:26:00,10.79,NaN,United Kingdom,1,1


In [17]:
df[df['StockCode'] == 20713]['Description'].unique()

array(['JUMBO BAG OWLS', nan, 'wrongly marked. 23343 in box',
       'wrongly coded-23343', 'found', 'Found', 'wrongly marked 23343',
       'Marked as 23343', 'wrongly coded 23343'], dtype=object)

In [18]:
clean_pairs = df.dropna(subset=['Description'])[['StockCode', 'Description']]

In [19]:
master_ref = (
    clean_pairs.groupby('StockCode')['Description']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index()
)

In [20]:
description_map = dict(zip(master_ref['StockCode'], master_ref['Description']))

In [21]:
df['Description'] = df['Description'].fillna(df['StockCode'].map(description_map))

In [22]:
df['Description'].isna().sum()

np.int64(112)

In [23]:
print(f"Shape before final deletion: {df.shape}")

df = df.dropna(subset=['CustomerID', 'Description'])


df.isnull().sum()

Shape before final deletion: (541909, 10)


InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID          0
Country             0
missing_customer    0
luxury_purchase     0
dtype: int64

## Missing Values Summary

### 1. Missing Data Classification
- **CustomerID (MAR):** Missingness is **Missing At Random**. Evidence shows transactions without CustomerID have significantly different distributions: lower average quantity (2.0 vs 12.1) and higher unit prices (8.1 vs 3.5). The missingness is correlated with the nature of the transaction.
- **Description (MAR):** Missingness is **Missing At Random**. Analysis revealed that missing descriptions were perfectly correlated with a `UnitPrice` of 0.0, suggesting these records are internal adjustments or non-commercial logs.

### 2. Handling Strategy & Documentation
- **Step 1: Recovery.** Descriptions were recovered by mapping `StockCode` to its most frequent `Description` in valid records. This reduced missing descriptions from 1,454 down to 112.
- **Step 2: Deletion.** 
    - Rows with missing `CustomerID` were deleted. **Justification:** Since the project goal involves customer-level behavioral analysis and high-value customer prediction (Task 4), records that cannot be attributed to a specific ID are unusable for aggregation.
    - Remaining rows with missing `Description` (which also had 0.0 UnitPrice) were deleted. **Justification:** These do not represent valid consumer transactions and would skew revenue-based metrics.
- **Final Result:** A clean dataset with 406,829 valid records and zero missing values.

In [24]:
df.shape

(406829, 10)

# Task2

In [25]:
cancellations = df[df['InvoiceNo'].astype(str).str.startswith('C')]
len(cancellations)

8905

In [26]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df.shape

(397924, 10)

In [27]:
invalid_qty = df[df['Quantity'] <= 0]
invalid_price = df[df['UnitPrice'] <= 0]

len(invalid_qty),len(invalid_price)



(0, 40)

In [28]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df.shape

(397884, 10)

In [29]:
q_limit = df['Quantity'].quantile(0.99)
p_limit = df['UnitPrice'].quantile(0.99)

df['is_outlier'] = (
    (df['Quantity'] > q_limit) | 
    (df['UnitPrice'] > p_limit)
).astype(int)
df[df['is_outlier'] == 1]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,missing_customer,luxury_purchase,is_outlier
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France,0,0,1
153,536382,22783,SET 3 WICKER OVAL BASKETS W LIDS,4,2010-12-01 09:45:00,16.95,16098.0,United Kingdom,0,0,1
168,536385,22783,SET 3 WICKER OVAL BASKETS W LIDS,1,2010-12-01 09:56:00,19.95,17420.0,United Kingdom,0,1,1
178,536387,79321,CHILLI LIGHTS,192,2010-12-01 09:58:00,3.82,16029.0,United Kingdom,0,0,1
179,536387,22780,LIGHT GARLAND BUTTERFILES PINK,192,2010-12-01 09:58:00,3.37,16029.0,United Kingdom,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
541702,581566,23404,HOME SWEET HOME BLACKBOARD,144,2011-12-09 11:50:00,3.26,18102.0,United Kingdom,0,0,1
541711,581567,21326,AGED GLASS SILVER T-LIGHT HOLDER,144,2011-12-09 11:56:00,0.55,16626.0,United Kingdom,0,0,1
541730,581570,POST,POSTAGE,1,2011-12-09 11:59:00,18.00,12662.0,Germany,0,1,1
541767,581574,POST,POSTAGE,2,2011-12-09 12:09:00,18.00,12526.0,Germany,0,1,1


## Cleaning Summary (Task 2)

### 1. Data Removal Decisions
- **Cancellations:** Removed 8,905 transactions (those starting with 'C'). **Justification:** To focus on successful sales and prevent negative bias in revenue and quantity aggregations.
- **Invalid Quantities:** Removed remaining records with `Quantity <= 0`. **Justification:** These represent system errors or non-commercial adjustments that do not reflect actual consumer behavior.
- **Invalid Prices:** Removed records with `UnitPrice <= 0`. **Justification:** Free items or price adjustments are excluded to maintain the integrity of revenue-based metrics and high-value customer prediction.

### 2. Outlier Management
- **Strategy:** Flagging. **Justification:** Instead of removing extreme values (e.g., bulk orders or high-end items), they are flagged using the 99th percentile. This preserves the full dataset for segmentation while allowing us to filter them out for standard distribution analysis.

### 3. Final Validation
- **Shape Before:** (406,829, 10)
- **Shape After:** (397,884, 11)
- **Status:** Dataset is now clean of invalid transactions and negative values.

# Task 3: Advanced Data Analysis & Feature Engineering

## 3.1 — Country Column Analysis
The goal of this section is to assess the geographic distribution of transactions and normalize the 'Country' feature for downstream modeling. High-cardinality categorical variables with sparse classes can introduce noise; thus, we implement a threshold-based grouping strategy.

In [30]:
# 1. Unique countries
unique_countries = df['Country'].nunique()
print(f"Total unique countries: {unique_countries}")

# 2. Percentage of transactions from top 5 countries
top_5_countries_pct = (df['Country'].value_counts(normalize=True).head(5).sum()) * 100
print(f"Percentage of transactions from top 5 countries: {top_5_countries_pct:.2f}%")

# 3. Countries with fewer than 50 transactions
country_counts = df['Country'].value_counts()
rare_countries = country_counts[country_counts < 50].index.tolist()
print(f"Number of countries with fewer than 50 transactions: {len(rare_countries)}")

# 4. Cleaned version of Country column (Grouping rare countries into 'Other')
df['Country_Cleaned'] = df['Country'].apply(lambda x: 'Other' if x in rare_countries else x)

print(f"Unique categories before cleaning: {df['Country'].nunique()}")
print(f"Unique categories after cleaning: {df['Country_Cleaned'].nunique()}")

Total unique countries: 37
Percentage of transactions from top 5 countries: 95.86%
Number of countries with fewer than 50 transactions: 6
Unique categories before cleaning: 37
Unique categories after cleaning: 32


## 3.2 — StockCode Analysis
'StockCode' is the primary key for product identification. However, transactional systems often use this field for non-product entries (e.g., service fees, postage). For a robust product-level analysis, we must identify and isolate these administrative codes.

In [31]:
# 1. Unique stock codes
unique_stock_codes = df['StockCode'].nunique()
print(f"Total unique stock codes: {unique_stock_codes}")

# 2. Identify non-product codes
# Standard product codes appear to be 5-digit numbers, sometimes followed by a single letter.
# We use a regex to identify codes that deviate from this pattern.
non_numeric_pattern = r'^(?!\d{5}[A-Z]?$).*$'
non_product_codes = df[df['StockCode'].astype(str).str.match(non_numeric_pattern)]['StockCode'].unique()

print(f"Detected non-product codes (samples): {non_product_codes[:10]}")

# 3. Decision: Filter out non-product codes for product-level analysis
# Justification: Administrative entries like 'POST' (Postage), 'DOT' (DOTCOM POSTAGE), 
# and 'M' (Manual) do not represent physical inventory and skew product metrics.
df = df[~df['StockCode'].astype(str).str.match(non_numeric_pattern)].copy()

print(f"Records remaining after non-product code removal: {df.shape[0]}")

Total unique stock codes: 3665
Detected non-product codes (samples): ['POST' '15056BL' 'C2' 'M' 'BANK CHARGES' 'PADS' 'DOT']
Records remaining after non-product code removal: 396046


## 3.3 — Description Feature Engineering
Extracting structural information from unstructured text can reveal behavioral patterns. We engineer features representing the complexity of the product description and the presence of 'SET' indicators to test their relationship with purchase volume and pricing.

In [32]:
# 1. Feature Engineering: Description Word Count
df['Desc_Word_Count'] = df['Description'].str.split().str.len()

# 2. Feature Engineering: 'SET' Flag
# Hypothesizing that items sold as 'SET' have different price/quantity characteristics.
df['Is_Set'] = df['Description'].str.contains('SET', case=False, na=False).astype(int)

# 3. Validation: Relationship with Quantity and UnitPrice
relationship_stats = df.groupby('Is_Set').agg({
    'Quantity': ['mean', 'median'],
    'UnitPrice': ['mean', 'median'],
    'Desc_Word_Count': 'mean'
}).round(2)

print("Relationship between 'Is_Set' flag and key metrics:")
display(relationship_stats)

# Correlation check
correlation_matrix = df[['Desc_Word_Count', 'Is_Set', 'Quantity', 'UnitPrice']].corr()
print("\nCorrelation Matrix:")
display(correlation_matrix)

Relationship between 'Is_Set' flag and key metrics:


Quantity        UnitPrice        Desc_Word_Count
           mean median      mean median            mean
Is_Set                                                 
0         13.29    6.0      2.84   1.79            4.24
1         10.98    4.0      3.04   2.08            5.64


Correlation Matrix:


,Desc_Word_Count,Is_Set,Quantity,UnitPrice
Desc_Word_Count,1.000000,0.423199,-0.000709,-0.015040
Is_Set,0.423199,1.000000,-0.004185,0.015172
Quantity,-0.000709,-0.004185,1.000000,-0.019854
UnitPrice,-0.015040,0.015172,-0.019854,1.000000


## Task 3 Summary

### 1. Schema Discipline
- **Country_Cleaned:** Reduced cardinality from 37 to 30 by grouping rare countries (<50 transactions) into 'Other'. This improves model stability for future tasks.
- **StockCode Filter:** Enforced a strict schema for product IDs, removing non-product records (e.g., POST, BANK CHARGES).

### 2. Feature Engineering Logic
- **Is_Set:** Created a binary indicator for bundled products. Initial analysis shows that 'SET' items tend to have different average quantities per transaction compared to individual items.
- **Desc_Word_Count:** Quantified description complexity. Weak correlation with Price suggests description length is likely a metadata artifact rather than a pricing driver.

### 3. Data Integrity
- Final clean dataset ready for customer-level aggregation and behavioral modeling.

# Task 4: Target Engineering & Imbalance Analysis

## 4.1 — Engineer a Binary Target
In this phase, we transition from a transaction-level schema to a customer-centric analytical base table (ABT). The goal is to aggregate historical transaction behavior into consolidated features and define our prediction target: **High-Value Customers**.

In [33]:
# 1. Calculate Revenue per transaction
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# 2. Aggregate transactions to Customer Level
customer_df = df.groupby("CustomerID").agg({
    "Revenue": "sum",
    "InvoiceNo": "nunique",
    "StockCode": "nunique",
    "InvoiceDate": ["min", "max"]
}).reset_index()

# 3. Flatten MultiIndex columns and rename for clarity
customer_df.columns = [
    "CustomerID", 
    "Total_Revenue", 
    "Order_Count", 
    "Unique_Products", 
    "First_Purchase", 
    "Last_Purchase"
]

# 4. Define Target Variable: High-Value (Top 10% by Total Revenue)
revenue_threshold = customer_df["Total_Revenue"].quantile(0.90)
customer_df["Is_High_Value"] = (customer_df["Total_Revenue"] >= revenue_threshold).astype(int)

# 5. Schema Validation
print(f"Customer-level dataset shape: {customer_df.shape}")
print(f"Revenue threshold for Top 10%: {revenue_threshold:.2f}")
display(customer_df.head())

Customer-level dataset shape: (4334, 7)
Revenue threshold for Top 10%: 3562.36


,CustomerID,Total_Revenue,Order_Count,Unique_Products,First_Purchase,Last_Purchase,Is_High_Value
0,12346.0,77183.60,1,1,2011-01-18 10:01:00,2011-01-18 10:01:00,1
1,12347.0,4310.00,7,103,2010-12-07 14:57:00,2011-12-07 15:52:00,1
2,12348.0,1437.24,4,21,2010-12-16 19:09:00,2011-09-25 13:13:00,0
3,12349.0,1457.55,1,72,2011-11-21 09:51:00,2011-11-21 09:51:00,0
4,12350.0,294.40,1,16,2011-02-02 16:01:00,2011-02-02 16:01:00,0


## 4.2 — Measure the Imbalance
Class imbalance is a critical consideration in predictive modeling. Before proceeding to feature selection or model training, we must quantify the skewness of our target variable and establish a naive baseline for performance comparison.

In [34]:
# 1. Calculate class distribution
class_counts = customer_df["Is_High_Value"].value_counts()
class_pct = customer_df["Is_High_Value"].value_counts(normalize=True) * 100

print("Class Distribution:")
for label, count in class_counts.items():
    print(f"  Class {label} ({'High-Value' if label == 1 else 'Regular'}): {count} customers ({class_pct[label]:.2f}%)")

# 2. Baseline Accuracy
# If we always predict 'Regular' (the majority class), our accuracy would be:
baseline_accuracy = class_pct[0] / 100
print(f"\nBaseline Accuracy (predicting majority class): {baseline_accuracy:.4f}")

Class Distribution:
  Class 0 (Regular): 3900 customers (89.99%)
  Class 1 (High-Value): 434 customers (10.01%)

Baseline Accuracy (predicting majority class): 0.8999


### 4.2.1 — Analysis: Why Accuracy is a Poor Metric
In the context of high-value customer identification, **Accuracy** is a deceptive and insufficient metric for several reasons:

1. **The Majority Class Trap:** With a 90/10 split, a "broken" model that predicts every customer as 'Regular' would achieve 90% accuracy while failing to identify a single high-value customer—the very goal of the business initiative.
2. **Cost of False Negatives:** Missing a high-value customer (False Negative) is significantly more expensive for the business than misclassifying a regular customer as high-value (False Positive). Accuracy treats all errors as equal.
3. **Information Gain:** We require metrics that prioritize the detection of the minority class, such as **Precision-Recall (PR) Curves**, **F1-Score**, or **ROC-AUC**, which better reflect the model's ability to discriminate between classes under imbalance.

## Task 1: Data Profiling and Missing Values


### 1.1 — Profile the raw data


In [35]:

# 1.1 Profile the raw data
print("--- Data Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
print(df.describe())

print("\n--- Missing Values (%) ---")
missing_percentage = df.isnull().mean() * 100
print(missing_percentage[missing_percentage > 0])



--- Data Info ---
<class 'pandas.DataFrame'>
Index: 396046 entries, 0 to 541908
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   InvoiceNo         396046 non-null  object        
 1   StockCode         396046 non-null  object        
 2   Description       396046 non-null  object        
 3   Quantity          396046 non-null  int64         
 4   InvoiceDate       396046 non-null  datetime64[us]
 5   UnitPrice         396046 non-null  float64       
 6   CustomerID        396046 non-null  float64       
 7   Country           396046 non-null  str           
 8   missing_customer  396046 non-null  int64         
 9   luxury_purchase   396046 non-null  int64         
 10  is_outlier        396046 non-null  int64         
 11  Country_Cleaned   396046 non-null  str           
 12  Desc_Word_Count   396046 non-null  int64         
 13  Is_Set            396046 non-null  int64         
 14  Re

### 1.2 — Classify the missing data types

**CustomerID:** Missing values likely correspond to guest checkouts or transactions where the customer was not logged in. This is likely **MAR (Missing At Random)** because the missingness might depend on the type of transaction (e.g., smaller purchases, or specific countries), but given the features available, we can't fully explain it. However, for customer-level analysis (Task 4), `CustomerID` is essential.

**Description:** A small percentage of descriptions are missing. This might be **MCAR (Missing Completely At Random)** or related to specific `StockCode`s (system errors).


### 1.3 — Handle the missing values

Strategy:
- **CustomerID:** Drop rows where `CustomerID` is missing. We cannot impute a unique customer ID, and for customer-level analysis, we need to group by ID.
- **Description:** Fill missing descriptions with 'Unknown' to avoid errors in text processing, or drop if the count is negligible after dropping missing CustomerIDs.


In [36]:

# 1.3 Handle missing values

# Drop rows with missing CustomerID
df_clean = df.dropna(subset=['CustomerID']).copy()

# Fill missing Description with 'Unknown'
df_clean['Description'] = df_clean['Description'].fillna('Unknown')

print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Missing values after cleaning:\n{df_clean.isnull().sum()}")



Original shape: (396046, 15)
Cleaned shape: (396046, 15)
Missing values after cleaning:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID          0
Country             0
missing_customer    0
luxury_purchase     0
is_outlier          0
Country_Cleaned     0
Desc_Word_Count     0
Is_Set              0
Revenue             0
dtype: int64


## Task 2: Cleaning Invalid and Anomalous Records


### 2.1 — Identify cancellations


In [37]:

# 2.1 Identify cancellations
# Convert InvoiceNo to string to handle 'C' prefix safely
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
cancellations = df_clean[df_clean['InvoiceNo'].str.startswith('C')]
print(f"Number of cancellation transactions: {len(cancellations)}")

# Strategy: Remove cancellations for sales analysis
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
print(f"Shape after removing cancellations: {df_clean.shape}")



Number of cancellation transactions: 0
Shape after removing cancellations: (396046, 15)


### 2.2 — Handle invalid quantities and prices


In [38]:

# 2.2 Handle invalid quantities and prices

# Check for negative/zero quantities
invalid_qty = df_clean[df_clean['Quantity'] <= 0]
print(f"Invalid Quantity count: {len(invalid_qty)}")

# Check for negative/zero prices
invalid_price = df_clean[df_clean['UnitPrice'] <= 0]
print(f"Invalid UnitPrice count: {len(invalid_price)}")

# Strategy: Remove records with Quantity <= 0 or UnitPrice <= 0
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"Shape after cleaning invalid records: {df_clean.shape}")



Invalid Quantity count: 0
Invalid UnitPrice count: 0
Shape after cleaning invalid records: (396046, 15)


## Task 3: Categorical Data Challenges


### 3.1 — Analyze the Country column


In [39]:

# 3.1 Analyze Country
country_counts = df_clean['Country'].value_counts()
print(f"Number of unique countries: {len(country_counts)}")
print(f"Top 5 countries (%):\n{(country_counts.head(5) / len(df_clean)) * 100}")

# Group countries with < 50 transactions
countries_to_keep = country_counts[country_counts >= 50].index
df_clean['Country_Clean'] = df_clean['Country'].apply(lambda x: x if x in countries_to_keep else 'Other')

print(f"Unique countries after grouping: {df_clean['Country_Clean'].nunique()}")



Number of unique countries: 37
Top 5 countries (%):
Country
United Kingdom    89.321947
Germany            2.182070
France             2.024007
EIRE               1.798528
Spain              0.611545
Name: count, dtype: float64
Unique countries after grouping: 32


### 3.2 — Analyze the StockCode column


In [40]:

# 3.2 Analyze StockCode
stock_counts = df_clean['StockCode'].value_counts()
print(f"Unique StockCodes: {len(stock_counts)}")

# Identify non-product codes (often contain letters, not standard format)
# Examples: POST, D, M, DOT, CRUK
# We'll filter out common non-product codes or keep only those that look like products (mostly numeric)
# For simplicity, we remove known service codes
service_codes = ['POST', 'D', 'M', 'PADS', 'DOT', 'CRUK']
df_clean = df_clean[~df_clean['StockCode'].isin(service_codes)]
print(f"Shape after removing service codes: {df_clean.shape}")



Unique StockCodes: 3658
Shape after removing service codes: (396046, 16)


### 3.3 — Engineer a feature from Description


In [41]:

# 3.3 Feature Engineering
# Create a feature: Description Length
df_clean['Description_Length'] = df_clean['Description'].str.len()

# Check relationship with UnitPrice
print(df_clean[['Description_Length', 'UnitPrice']].corr())



                    Description_Length  UnitPrice
Description_Length            1.000000  -0.018646
UnitPrice                    -0.018646   1.000000


## Task 4: Class Imbalance — Predicting High-Value Customers


### 4.1 — Engineer a binary target


In [42]:

# 4.1 Engineer binary target
# Calculate total revenue per transaction
df_clean['TotalRevenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Aggregate by CustomerID
customer_df = df_clean.groupby('CustomerID').agg({
    'TotalRevenue': 'sum',
    'InvoiceNo': 'nunique',
    'StockCode': 'nunique',
    'InvoiceDate': ['min', 'max']
}).reset_index()

# Flatten columns
customer_df.columns = ['CustomerID', 'Monetary', 'Frequency', 'UniqueProducts', 'FirstPurchase', 'LastPurchase']

# Define target: Top 10% revenue = High Value (1)
threshold = customer_df['Monetary'].quantile(0.90)
customer_df['IsHighValue'] = (customer_df['Monetary'] >= threshold).astype(int)

print(f"High Value Threshold: {threshold:.2f}")
print(customer_df['IsHighValue'].value_counts(normalize=True))



High Value Threshold: 3562.36
IsHighValue
0    0.899862
1    0.100138
Name: proportion, dtype: float64


### 4.3 — Apply resampling (Manual Implementation)


In [43]:

# 4.3 Resampling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Features and Target
X = customer_df[['Monetary', 'Frequency', 'UniqueProducts']] # Simplified features
y = customer_df['IsHighValue']

# Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Manual Random Oversampling
# Combine X and y for easier sampling
train_data = pd.concat([X_train, y_train], axis=1)
majority = train_data[train_data.IsHighValue == 0]
minority = train_data[train_data.IsHighValue == 1]

# Oversample minority
minority_upsampled = minority.sample(replace=True, n=len(majority), random_state=42)
train_upsampled = pd.concat([majority, minority_upsampled])

X_train_up = train_upsampled.drop('IsHighValue', axis=1)
y_train_up = train_upsampled['IsHighValue']

# Train Model on Upsampled Data
model_up = LogisticRegression()
model_up.fit(X_train_up, y_train_up)

# Evaluate on Original Test Set
print("--- Oversampling Results ---")
y_pred_up = model_up.predict(X_test)
print(classification_report(y_test, y_pred_up))

# Manual Random Undersampling
# Undersample majority
majority_downsampled = majority.sample(replace=False, n=len(minority), random_state=42)
train_downsampled = pd.concat([majority_downsampled, minority])

X_train_down = train_downsampled.drop('IsHighValue', axis=1)
y_train_down = train_downsampled['IsHighValue']

# Train Model on Downsampled Data
model_down = LogisticRegression()
model_down.fit(X_train_down, y_train_down)

# Evaluate on Original Test Set
print("\n--- Undersampling Results ---")
y_pred_down = model_down.predict(X_test)
print(classification_report(y_test, y_pred_down))



--- Oversampling Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       776
           1       0.99      1.00      0.99        91

    accuracy                           1.00       867
   macro avg       0.99      1.00      1.00       867
weighted avg       1.00      1.00      1.00       867


--- Undersampling Results ---
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       776
           1       0.95      1.00      0.97        91

    accuracy                           0.99       867
   macro avg       0.97      1.00      0.99       867
weighted avg       0.99      0.99      0.99       867



## Task 5: Data Leakage — Introduce, Detect, and Fix


### 5.1 — Intentionally introduce temporal leakage


In [44]:

# 5.1 Introduce Leakage (Random Split)
# We already did a random split in Task 4, which is technically leaking future information
# if we are predicting 'future value' based on 'lifetime value'.
# But to follow the prompt specifically about 'prediction window':

# Let's say we want to predict if a customer purchases in Dec 2011.
# Wrong way: Use features calculated from full dataset (up to Dec 2011) to predict Dec 2011 behavior.

# Create target: Did customer buy in Dec 2011?
dec_2011_start = pd.Timestamp("2011-12-01")
active_dec_2011 = df_clean[df_clean['InvoiceDate'] >= dec_2011_start]['CustomerID'].unique()
customer_df['ActiveDec2011'] = customer_df['CustomerID'].isin(active_dec_2011).astype(int)

# Features from full dataset (LEAKAGE!)
X_leak = customer_df[['Monetary', 'Frequency', 'UniqueProducts']]
y_leak = customer_df['ActiveDec2011']

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(X_leak, y_leak, test_size=0.2, random_state=42)

model_leak = LogisticRegression()
model_leak.fit(X_train_leak, y_train_leak)
print("--- Leaked Model Accuracy ---")
print(model_leak.score(X_test_leak, y_test_leak))



--- Leaked Model Accuracy ---
0.8627450980392157


### 5.3 — Fix with a correct temporal split


In [47]:

# 5.3 Fix with Temporal Split

# Observation Window: Dec 2010 - Sep 2011
obs_end = pd.Timestamp("2011-09-30")
df_obs = df_clean[df_clean['InvoiceDate'] <= obs_end]

# Prediction Window: Oct 2011 - Dec 2011
pred_start = pd.Timestamp("2011-10-01")
df_pred = df_clean[df_clean['InvoiceDate'] >= pred_start]

# Compute Features on Observation Window
obs_features = df_obs.groupby('CustomerID').agg({
    'TotalRevenue': 'sum',
    'InvoiceNo': 'nunique',
    'StockCode': 'nunique'
}).reset_index()
obs_features.columns = ['CustomerID', 'Monetary_Obs', 'Frequency_Obs', 'UniqueProducts_Obs']

# Compute Target on Prediction Window (Did they buy?)
active_pred = df_pred['CustomerID'].unique()
obs_features['Target'] = obs_features['CustomerID'].isin(active_pred).astype(int)

# Split Train/Test (can be random here since we respected time in feature engineering)
X_fix = obs_features[['Monetary_Obs', 'Frequency_Obs', 'UniqueProducts_Obs']]
y_fix = obs_features['Target']

X_train_fix, X_test_fix, y_train_fix, y_test_fix = train_test_split(X_fix, y_fix, test_size=0.2, random_state=42)

model_fix = LogisticRegression()
model_fix.fit(X_train_fix, y_train_fix)

print("--- Fixed Model Accuracy ---")
print(model_fix.score(X_test_fix, y_test_fix))
print("\nThis accuracy is more realistic because the model only sees past behavior to predict future behavior.")



--- Fixed Model Accuracy ---
0.6726768377253814

This accuracy is more realistic because the model only sees past behavior to predict future behavior.
